# Entraînement du modèle « bon état / endommagé »

Premier modèle de la version 2. Il répond à une seule question : **la pièce est-elle endommagée ?**

Données : `data/v2/` (voir `preparation_donnees_v2.ipynb`). Modèle : ResNet18 préentraîné sur ImageNet, comme en version 1.

Sections :
1. chargement des données;
2. entraînement (environ 50 minutes, images en 384 pixels);
3. évaluation sur le jeu de test, au global et par source;
4. erreurs du modèle;
5. zones utilisées par le modèle (Grad-CAM);
6. test sur les photos réelles.

Cliquez sur **Run All**.

In [ ]:
import json
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, precision_score, recall_score
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms

RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RACINE))
from diagnostic.vision import ECART_TYPE, MODELE_ETAT, MOYENNE, creer_resnet, grad_cam, transformation_etat

DONNEES = RACINE / "data" / "v2"
CHEMIN_MODELE = MODELE_ETAT
DOSSIER_RESULTATS = RACINE / "resultats"
GRAINE = 42
EPOQUES = 10
TAILLE = 384  # pixels : les défauts fins (vis, écrous) restent visibles

torch.manual_seed(GRAINE)
random.seed(GRAINE)
plt.rcParams["figure.dpi"] = 110
NOMS = {"bon_etat": "Bon état", "endommage": "Endommagé"}
print("PyTorch", torch.__version__)

## 1. Chargement des données

L'augmentation de données prépare le modèle à de vraies photos : recadrages, rotations,
variations de luminosité, et passage en niveaux de gris une fois sur cinq (les photos Casting sont en gris).

Deux choix pour les petites sources aux défauts fins (écrous et vis de MVTec) :
- **recadrage léger** (au moins 85 % de l'image) : un recadrage plus fort peut couper un petit défaut,
  et l'image « endommagée » ne montre alors plus aucun défaut;
- **tirage équilibré** : chaque groupe source × classe est tiré aussi souvent que les autres pendant l'entraînement,
  pour que les 49 écrous endommagés ne soient pas noyés parmi les 4 600 autres images.
- **images en 384 pixels** au lieu de 224 : un filet de vis écrasé reste visible après la réduction de l'image.

In [ ]:
transfo_entrainement = transforms.Compose([
    transforms.RandomResizedCrop(TAILLE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MOYENNE, ECART_TYPE),
])
transfo_evaluation = transformation_etat(TAILLE)  # même transformation que l'application (diagnostic/vision.py)

jeux = {
    "train": datasets.ImageFolder(DONNEES / "train", transform=transfo_entrainement),
    "val": datasets.ImageFolder(DONNEES / "val", transform=transfo_evaluation),
    "test": datasets.ImageFolder(DONNEES / "test", transform=transfo_evaluation),
}
CLASSES = jeux["train"].classes
# Groupe de chaque image : source (début du nom de fichier) et classe
groupes = {nom: [(Path(chemin).name.split("_")[0], CLASSES[cible]) for chemin, cible in jeu.samples]
           for nom, jeu in jeux.items()}
effectifs = pd.Series(groupes["train"]).value_counts()
poids = [1.0 / effectifs[g] for g in groupes["train"]]
tirage = WeightedRandomSampler(poids, num_samples=len(poids), replacement=True,
                               generator=torch.Generator().manual_seed(GRAINE))
chargeurs = {
    "train": DataLoader(jeux["train"], batch_size=32, sampler=tirage, num_workers=0),
    "val": DataLoader(jeux["val"], batch_size=32, num_workers=0),
    "test": DataLoader(jeux["test"], batch_size=32, num_workers=0),
}

print("Classes :", CLASSES)
pd.DataFrame({nom: pd.Series([CLASSES[c] for c in jeu.targets]).value_counts() for nom, jeu in jeux.items()})

## 2. Entraînement

Le meilleur modèle est choisi sur la validation avec l'**exactitude équilibrée** : la moyenne des exactitudes
de chaque groupe source × classe. Ainsi, une petite source compte autant qu'une grande.

In [ ]:
modele = creer_resnet(len(CLASSES))

optimiseur = torch.optim.AdamW(modele.parameters(), lr=3e-4, weight_decay=1e-4)
planificateur = torch.optim.lr_scheduler.CosineAnnealingLR(optimiseur, T_max=EPOQUES)
perte_fn = nn.CrossEntropyLoss()

def exactitude_equilibree(nom):
    """Moyenne des exactitudes de chaque groupe source × classe (la validation n'est pas mélangée)."""
    modele.eval()
    corrects = []
    with torch.no_grad():
        for x, y in chargeurs[nom]:
            corrects += (modele(x).argmax(1) == y).tolist()
    return pd.Series(corrects).groupby(pd.Series(groupes[nom])).mean().mean()

historique, meilleure = [], 0.0
CHEMIN_MODELE.parent.mkdir(exist_ok=True)
for epoque in range(1, EPOQUES + 1):
    debut = time.time()
    modele.train()
    perte_totale = 0.0
    for i, (x, y) in enumerate(chargeurs["train"], 1):
        optimiseur.zero_grad()
        perte = perte_fn(modele(x), y)
        perte.backward()
        optimiseur.step()
        perte_totale += perte.item()
        print(f"\r  Époque {epoque}/{EPOQUES}  lot {i}/{len(chargeurs['train'])}", end="")
    planificateur.step()
    exact_val = exactitude_equilibree("val")
    historique.append({"epoque": epoque, "perte": perte_totale / len(chargeurs["train"]), "exactitude_validation": exact_val})
    print(f"\r  Époque {epoque}/{EPOQUES}  perte {historique[-1]['perte']:.3f}  "
          f"exactitude équilibrée validation {exact_val:.1%}  ({time.time() - debut:.0f} s)")
    if exact_val >= meilleure:
        meilleure = exact_val
        torch.save({"etat": modele.state_dict(), "classes": CLASSES, "epoque": epoque, "taille": TAILLE}, CHEMIN_MODELE)

print(f"\nMeilleure exactitude équilibrée en validation : {meilleure:.1%}  →  modèle enregistré dans {CHEMIN_MODELE.name}")
_ = (DOSSIER_RESULTATS / "v2_historique.json").write_text(json.dumps(historique, indent=2), encoding="utf-8")

In [ ]:
h = pd.DataFrame(historique)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(h["epoque"], h["perte"], marker="o")
ax1.set(title="Perte d'entraînement", xlabel="Époque", ylabel="Perte")
ax2.plot(h["epoque"], h["exactitude_validation"] * 100, marker="o", color="tab:green")
ax2.set(title="Exactitude équilibrée en validation", xlabel="Époque", ylabel="%")
for ax in (ax1, ax2):
    ax.grid(alpha=0.3)
    ax.set_xticks(h["epoque"])
plt.tight_layout()
plt.show()

## 2b. Décision sur la validation

Le meilleur modèle est évalué **par source sur la validation**. C'est ce tableau qui décide si l'essai est retenu,
avant de regarder le test. Règle fixée d'avance : **chaque source atteint au moins 85 % d'exactitude en validation**.
Sinon, les écrous et les vis sont retirés du jeu (préparation), et le modèle est réentraîné une dernière fois.

In [ ]:
sauvegarde = torch.load(CHEMIN_MODELE, map_location="cpu", weights_only=True)
modele.load_state_dict(sauvegarde["etat"])
modele.eval()
predites_val = []
with torch.no_grad():
    for x, _ in chargeurs["val"]:
        predites_val += modele(x).argmax(1).tolist()
val = pd.DataFrame(groupes["val"], columns=["source", "vraie"])
val["correct"] = [CLASSES[p] == v for p, v in zip(predites_val, val["vraie"])]
decision = val.groupby("source").agg(Images=("correct", "size"), Exactitude=("correct", "mean"))
decision["Endommagées détectées"] = val[val["vraie"] == "endommage"].groupby("source")["correct"].mean()
retenu = bool((decision["Exactitude"] >= 0.85).all())
print(f"Modèle de l'époque {sauvegarde['epoque']} ({sauvegarde['taille']} px)")
print("ESSAI RETENU" if retenu else "ESSAI NON RETENU : retirer les écrous et les vis, puis réentraîner une dernière fois")
decision.style.format({"Exactitude": "{:.1%}", "Endommagées détectées": "{:.1%}"})

## 3. Évaluation sur le jeu de test

Le meilleur modèle est rechargé, puis évalué sur les images de test, jamais vues pendant l'entraînement.
La classe positive est **endommagé** : le rappel indique la part des pièces endommagées détectées,
la précision la part des alertes justifiées. Critères visés : rappel ≥ 90 % et précision ≥ 85 %.

In [ ]:
POSITIF = CLASSES.index("endommage")
vraies, predites, probas = [], [], []
with torch.no_grad():
    for x, y in chargeurs["test"]:
        p = torch.softmax(modele(x), dim=1)
        probas += p[:, POSITIF].tolist()
        predites += p.argmax(1).tolist()
        vraies += y.tolist()

test = pd.DataFrame({
    "fichier": [Path(chemin).name for chemin, _ in jeux["test"].samples],
    "chemin": [chemin for chemin, _ in jeux["test"].samples],
    "vraie": [CLASSES[v] for v in vraies],
    "predite": [CLASSES[p] for p in predites],
    "proba_endommage": probas,
})
test["source"] = test["fichier"].str.split("_").str[0]
test["correct"] = test["vraie"] == test["predite"]

def mesures(df):
    v = (df["vraie"] == "endommage").astype(int)
    p = (df["predite"] == "endommage").astype(int)
    return {"Images": len(df), "Exactitude": df["correct"].mean(),
            "Rappel (endommagé)": recall_score(v, p, zero_division=0),
            "Précision (endommagé)": precision_score(v, p, zero_division=0)}

resultats = pd.DataFrame({"Global": mesures(test),
                          **{f"Source : {s}": mesures(g) for s, g in test.groupby("source")}}).T
resultats["Images"] = resultats["Images"].astype(int)

rappel, precision = resultats.loc["Global", "Rappel (endommagé)"], resultats.loc["Global", "Précision (endommagé)"]
print(f"Rappel    : {rappel:.1%}  (critère ≥ 90 %) → {'ATTEINT' if rappel >= 0.90 else 'NON ATTEINT'}")
print(f"Précision : {precision:.1%}  (critère ≥ 85 %) → {'ATTEINT' if precision >= 0.85 else 'NON ATTEINT'}")

_ = (DOSSIER_RESULTATS / "v2_metriques_test.json").write_text(resultats.to_json(orient="index", indent=2), encoding="utf-8")
resultats.style.format({"Exactitude": "{:.1%}", "Rappel (endommagé)": "{:.1%}", "Précision (endommagé)": "{:.1%}"})

In [ ]:
panneaux = [("Global", test)] + [(f"Source : {s}", g) for s, g in test.groupby("source")]
colonnes = min(4, len(panneaux))
lignes_fig = int(np.ceil(len(panneaux) / colonnes))
fig, axes = plt.subplots(lignes_fig, colonnes, figsize=(4.6 * colonnes, 4.2 * lignes_fig), squeeze=False)
for ax in axes.flat[len(panneaux):]:
    ax.axis("off")
for ax, (titre, df) in zip(axes.flat, panneaux):
    matrice = confusion_matrix(df["vraie"], df["predite"], labels=CLASSES)
    ConfusionMatrixDisplay(matrice, display_labels=[NOMS[c] for c in CLASSES]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set(title=titre, xlabel="Prédit", ylabel="Réel")
plt.tight_layout()
fig.savefig(DOSSIER_RESULTATS / "v2_matrices_confusion.png", dpi=150)
plt.show()

## 4. Erreurs du modèle

Les images de test mal classées (12 au maximum), avec la probabilité « endommagé » donnée par le modèle.

In [ ]:
erreurs = test[~test["correct"]]
print(f"{len(erreurs)} erreur(s) sur {len(test)} images de test")
print(erreurs.groupby(["source", "vraie"]).size().rename("erreurs").to_string())
if len(erreurs):
    montrees = erreurs.sample(min(12, len(erreurs)), random_state=0)
    colonnes = min(4, len(montrees))
    lignes_fig = int(np.ceil(len(montrees) / colonnes))
    fig, axes = plt.subplots(lignes_fig, colonnes, figsize=(3.2 * colonnes, 3.4 * lignes_fig), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, e) in zip(axes.flat, montrees.iterrows()):
        ax.imshow(Image.open(e["chemin"]))
        ax.set_title(f"{e['source']} – réel : {NOMS[e['vraie']]}\nprédit : {NOMS[e['predite']]} "
                     f"({e['proba_endommage']:.0%} endommagé)", fontsize=8, color="red")
    plt.tight_layout()
    plt.show()

## 5. Où le modèle regarde-t-il ? (Grad-CAM)

Pour chaque source, une image en bon état et une image endommagée du jeu de test.
On vérifie que le modèle regarde la pièce et ses défauts, plutôt que le fond de l'image.

In [ ]:
def vue_modele(image):
    # Même cadrage que transfo_evaluation, sans normalisation, pour l'affichage
    return transforms.CenterCrop(TAILLE)(transforms.Resize(round(TAILLE * 256 / 224))(image))

choix = [g.sample(1, random_state=3).iloc[0] for _, g in test.groupby(["source", "vraie"])]
PAR_LIGNE = 6  # au-delà, les images passent à la ligne suivante
colonnes = min(PAR_LIGNE, len(choix))
blocs = int(np.ceil(len(choix) / colonnes))
fig, axes = plt.subplots(2 * blocs, colonnes, figsize=(3.3 * colonnes, 3.5 * 2 * blocs), squeeze=False)
for ax in axes.flat:
    ax.axis("off")
for k, e in enumerate(choix):
    haut, j = 2 * (k // colonnes), k % colonnes
    image = Image.open(e["chemin"]).convert("RGB")
    carte, classe = grad_cam(modele, transfo_evaluation(image).unsqueeze(0))
    axes[haut, j].imshow(vue_modele(image))
    axes[haut, j].set_title(f"{e['source']} – {NOMS[e['vraie']]}", fontsize=9)
    axes[haut + 1, j].imshow(vue_modele(image))
    axes[haut + 1, j].imshow(carte, cmap="jet", alpha=0.45)
    axes[haut + 1, j].set_title(f"Prédit : {NOMS[CLASSES[classe]]}", fontsize=9)
fig.suptitle("Images de test (haut) et zones utilisées par le modèle (bas)", fontsize=12)
plt.tight_layout()
fig.savefig(DOSSIER_RESULTATS / "v2_grad_cam.png", dpi=130)
plt.show()

## 6. Test sur les photos réelles

Les photos de `data/photos_reelles/` : le dossier `aucun` correspond à une pièce en **bon état**,
tous les autres dossiers (`rayures`, `piqures`…) à une pièce **endommagée**.

In [ ]:
DOSSIER_PHOTOS = RACINE / "data" / "photos_reelles"
photos = [p for p in sorted(DOSSIER_PHOTOS.rglob("*")) if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
print(f"{len(photos)} photo(s) trouvée(s)")

lignes = []
for chemin in photos:
    dossier = chemin.parent.name if chemin.parent != DOSSIER_PHOTOS else "inconnu"
    attendu = "bon_etat" if dossier == "aucun" else ("endommage" if dossier != "inconnu" else None)
    image = ImageOps.exif_transpose(Image.open(chemin)).convert("RGB")
    with torch.no_grad():
        p = torch.softmax(modele(transfo_evaluation(image).unsqueeze(0)), dim=1)[0, POSITIF].item()
    predit = "endommage" if p >= 0.5 else "bon_etat"
    lignes.append({"Photo": chemin.name, "Dossier": dossier, "Attendu": NOMS.get(attendu, "?"),
                   "Prédit": NOMS[predit], "Proba endommagé": p, "Correct": (predit == attendu) if attendu else None,
                   "chemin": chemin})
reelles = pd.DataFrame(lignes)

if len(reelles):
    evaluees = reelles.dropna(subset=["Correct"])
    print(f"Photos correctement classées : {int(evaluees['Correct'].sum())}/{len(evaluees)}")
    colonnes = min(8, len(reelles))
    lignes_fig = int(np.ceil(len(reelles) / colonnes))
    fig, axes = plt.subplots(lignes_fig, colonnes, figsize=(3 * colonnes, 3.6 * lignes_fig), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (_, r) in zip(axes.flat, reelles.iterrows()):
        ax.imshow(vue_modele(ImageOps.exif_transpose(Image.open(r["chemin"])).convert("RGB")))
        couleur = "black" if r["Correct"] is None else ("green" if r["Correct"] else "red")
        ax.set_title(f"{r['Dossier']}\n{r['Prédit']} ({r['Proba endommagé']:.0%})", fontsize=8, color=couleur)
    plt.tight_layout()
    fig.savefig(DOSSIER_RESULTATS / "v2_photos_reelles.png", dpi=130)
    plt.show()
    display(reelles.drop(columns="chemin").style.format({"Proba endommagé": "{:.0%}"}).hide(axis="index"))